In [ ]:
# ==============================================================================
# 2026改修版: SVM_Linear (設計基準 A〜K 完全準拠)
# - 重要度: svmLinearのネイティブ varImp を適用
# ==============================================================================

suppressPackageStartupMessages({
  library(caret)
  library(dplyr)
  library(Metrics)
  library(kernlab)
})

set.seed(42)

# -------------------------------
# (F)(G) 設定：パスと対象ファイル
# -------------------------------
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
file_names <- c(
  "n_base.csv", "n_base_OH_rebuilt.csv", "n_base_FP_rebuilt.csv",
  "n_all.csv",  "n_all_OH_rebuilt.csv",  "n_all_FP_rebuilt.csv",
  "m_base.csv", "m_base_OH_rebuilt.csv", "m_base_FP_rebuilt.csv",
  "m_all.csv",  "m_all_OH_rebuilt.csv",  "m_all_FP_rebuilt.csv"
)

# 出力先設定 (F)
today <- format(Sys.Date(), "%Y%m%d")
out_root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results"
model_name <- "SVM_Linear"
run_dir <- file.path(out_root, paste0("Corr_1000_", model_name))
if (!dir.exists(run_dir)) dir.create(run_dir, recursive = TRUE)

# (C)(D)(E) 除外変数設定
inappropriate_vars <- c(
  'Jsccal', 'JscJsccal', 'PCEaTA', 'PCEaTAFinal', 'EQEABEST', 'Rshthin', 'Dresistance',
  'mobilityeblendOFET', 'mobilityep1MOFET', 'mobilityhblendSCLC', 'mobilityhnMSCLC', 'mobilityhp1MSCLC',
  'IonIoffF', 'DRMS', 'Var.1'
)
physical_exclude_patterns <- c("X2DGIWAXD", "X2DGIXD", "widthfibril")

# -------------------------------
# HELPERS
# -------------------------------
safe_r2 <- function(y, p) {
  r <- suppressWarnings(cor(y, p))
  if (is.na(r)) return(NA_real_)
  return(as.numeric(r^2))
}

# -------------------------------
# MAIN LOOP
# -------------------------------
summary_list <- list()
importance_list <- list()

for (file in file_names) {
  filepath <- file.path(base_path, file)
  if (!file.exists(filepath)) next
  
  df_raw <- tryCatch(read.csv(filepath, row.names = 1, check.names = FALSE), error = function(e) NULL)
  if (is.null(df_raw)) next
  if ("X" %in% colnames(df_raw)) df_raw$X <- NULL

  for (target in target_vars) {
    if (!target %in% colnames(df_raw)) next

    # クレンジング
    df_temp <- df_raw[, sapply(df_raw, is.numeric)] %>% na.omit()
    if (nrow(df_temp) < 20) next

    # (C)(D)(E) 変数除外
    features <- setdiff(colnames(df_temp), target_vars)
    features <- setdiff(features, inappropriate_vars)
    features <- features[!grepl(paste(physical_exclude_patterns, collapse="|"), features)]
    
    # ゼロ分散除外
    vars <- sapply(df_temp[, features, drop = FALSE], var)
    features <- names(vars)[vars > 0 & !is.na(vars)]

    # (H) 多重共線性対策 (Corr > 0.99999)
    if (length(features) > 1) {
      cor_mat <- cor(df_temp[, features, drop = FALSE], use = "pairwise.complete.obs")
      cor_mat[is.na(cor_mat)] <- 0
      high_corr <- findCorrelation(cor_mat, cutoff = 0.99999)
      if (length(high_corr) > 0) features <- features[-high_corr]
    }
    if (length(features) == 0) next

    # (I) スケーリング [-1, 1]
    pp <- preProcess(df_temp[, features, drop = FALSE], method = "range")
    df_scaled <- df_temp
    df_scaled[, features] <- predict(pp, df_temp[, features]) * 2 - 1

    # (K) CV10設定 (A)OOD不要
    train_ctrl <- trainControl(method = "cv", number = 10, savePredictions = "final")
    tune_grid <- expand.grid(C = c(0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100))

    # 学習
    model <- tryCatch(
      train(
        x = df_scaled[, features, drop = FALSE], y = df_scaled[[target]],
        method = "svmLinear", trControl = train_ctrl, tuneGrid = tune_grid, metric = "RMSE"
      ),
      error = function(e) { cat("  [ERROR] SVM_Linear failed:", e$message, "\n"); return(NULL) }
    )
    if (is.null(model)) next

    # --- 結果保存 ---
    tag <- paste0(tools::file_path_sans_ext(file), "_", target)
    
    # 1. モデルRDS
    saveRDS(model, file.path(run_dir, paste0(tag, "_model.rds")))

    # 2. (B) CV10_OOF予測CSV
    pred_oof <- model$pred %>% 
      filter(C == model$bestTune$C) %>%
      mutate(SampleID = rownames(df_scaled)[rowIndex], Type = "CV10_OOF") %>%
      select(SampleID, Observed = obs, Predicted = pred, Type)
    write.csv(pred_oof, file.path(run_dir, paste0(tag, "_pred_OOF.csv")), row.names = FALSE)

    # 3. (J) 重要度 (svmLinearネイティブ varImp)
    # svmLinearの重要度は係数の重みに基づきます
    imp_val <- varImp(model, scale = FALSE)$importance
    imp_df <- data.frame(Feature = rownames(imp_val), Importance = imp_val$Overall)
    imp_df$File <- file
    imp_df$Target <- target
    importance_list[[length(importance_list)+1]] <- imp_df

    # 4. サマリー集計
    summary_list[[length(summary_list)+1]] <- data.frame(
      File=file, Target=target, 
      CV10_R2=safe_r2(pred_oof$Observed, pred_oof$Predicted),
      CV10_RMSE=rmse(pred_oof$Observed, pred_oof$Predicted), 
      n_samples=nrow(df_scaled),
      n_features=length(features), 
      Best_C=model$bestTune$C
    )
    
    cat(sprintf("  Finished: %s - %s\n", file, target))
  }
}

# CSV出力
if (length(summary_list) > 0) {
  write.csv(do.call(rbind, summary_list), file.path(run_dir, "all_summary_CV10.csv"), row.names = FALSE)
  write.csv(do.call(rbind, importance_list), file.path(run_dir, "all_importance_SVM.csv"), row.names = FALSE)
}

cat("\n[DONE] SVM_Linear Analysis completed.\n")

  Finished: n_base.csv - Jsc
  Finished: n_base.csv - Voc
  Finished: n_base.csv - FF
  Finished: n_base.csv - PCEmax


Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x,

  Finished: n_base_OH_rebuilt.csv - Jsc


Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x,

  Finished: n_base_OH_rebuilt.csv - Voc


...

Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x,

  Finished: m_all_OH_rebuilt.csv - Voc


Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x,

  Finished: m_all_OH_rebuilt.csv - FF


Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x,